**0. Data Import & Cleaning**

In [1]:
import os
import pandas as pd
import numpy as np

import re
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
nltk.download('stopwords')
default_stopwords = set(stopwords.words('english'))
medical_keywords = {"auto", "self", "up", "down", "p", "nf", "reg", "via", "path", "mod", "over", "under"}
custom_stopwords = default_stopwords - medical_keywords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/fiatlux/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df_raw = pd.read_csv('../data/processed/train_data.csv')
print(df_raw.shape)
df_raw.head()

(26250, 7)


,AC,PMID,Title,Abstract,Terms,Text_combined,batch_number
0,Q9W539,12537572,Annotation of the Drosophila melanogaster euch...,The recent completion of the Drosophila melano...,NaN,Annotation of the Drosophila melanogaster euch...,1
1,Q9FJI5,10437832,Evidence for functional convergence of redox r...,"In a recent paper (Wenderoth et al., J Biol Ch...",NaN,Evidence for functional convergence of redox r...,1
2,P0A382,2845100,Single amino acid changes in the Bacillus thur...,Site-directed mutagenesis has been used to cha...,NaN,Single amino acid changes in the Bacillus thur...,1
3,Q9LKL2,10926537,"Cloning of the Arabidopsis clock gene TOC1, an...",The toc1 mutation causes shortened circadian r...,autoregulation,"Cloning of the Arabidopsis clock gene TOC1, an...",1
4,Q96BR6,16710414,The DNA sequence and biological annotation of ...,The reference sequence for each human chromoso...,NaN,The DNA sequence and biological annotation of ...,1


In [4]:
# text cleaning 
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", "", text)  # Keep hyphens '-'
    text = " ".join([word for word in text.split() if word not in custom_stopwords])
    return text

# process 'Terms' column into list format
def process_terms_column(df: pd.DataFrame) -> pd.DataFrame:
    df["Terms"] = df["Terms"].fillna("non-autoregulatory")
    df["Terms_List"] = df["Terms"].apply(lambda x: [t.strip() for t in str(x).split(",")] if x and x.strip() else [])
    return df

# rule-based polarity inference function
def infer_polarity(text: str, term_list: list) -> str:
    text = text.lower()

    # negative regulation keywords
    negative_keywords = [
        'inhibit', 'repress', 'suppress', 'block', 'reduce', 'decrease', 'down-regulat', 
        'downregulat', 'negative', 'inactivat', 'stop', 'prevent',
        'attenuate', 'dampen', 'silence', 'knock', 'impair', 'abolish', 'diminish',
        'weaken', 'curtail', 'halt', 'terminate', 'cease', 'limit', 'restrict',
        'degradation', 'breakdown', 'turnover', 'cleavage', 'proteolysis',
        'downmodulat', 'counter', 'antagoniz', 'oppose'
    ]

    # positive regulation keywords
    positive_keywords = [
        'activat', 'increas', 'induce', 'enhance', 'promot', 'stimulat', 'up-regulat',
        'upregulat', 'positive', 'amplif', 'boost', 'augment', 'facilitate', 'accelerate',
        'catalyze', 'drive', 'trigger', 'elicit', 'evoke', 'potentiat', 'strengthen',
        'reinforce', 'foster', 'support', 'maintain', 'sustain', 'stabiliz', 'preserve',
        'accumul', 'recruit', 'upmodulat', 'elevat', 'heighten', 'agoniz'
    ]

    # count keyword appearances
    pos_count = sum([text.count(k) for k in positive_keywords])
    neg_count = sum([text.count(k) for k in negative_keywords])

    # Fallback mechanisms based on known mechanism names
    fallback_negative_terms = ["autoinhibition", "autorepression", "self-inhibition"]
    fallback_positive_terms = ["autoactivation", "self-activation"]

    # decision logic
    if pos_count > neg_count:
        return "positive"
    elif neg_count > pos_count:
        return "negative"
    elif any(t.lower() in fallback_negative_terms for t in term_list):
        return "negative"
    elif any(t.lower() in fallback_positive_terms for t in term_list):
        return "positive"
    else:
        return "neutral"

In [5]:
# main preprocessing pipeline
def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = process_terms_column(df)
    df["Text_Cleaned"] = df["Text_combined"].astype(str).apply(clean_text)

    # only infer polarity if the term is not solely "non-autoregulatory"
    def safe_infer(row):
        terms = [t.lower() for t in row["Terms_List"]]
        if len(terms) > 0 and not (len(terms) == 1 and terms[0] == "non-autoregulatory"):
            return infer_polarity(row["Text_Cleaned"], row["Terms_List"])
        else:
            return None

    df["polarity"] = df.apply(safe_infer, axis=1)
    return df[["batch_number", "Text_Cleaned", "Terms", "Terms_List", "polarity"]].copy()

# encode mechanism and polarity labels
def encode_labels(df_cleaned: pd.DataFrame):
    mlb = MultiLabelBinarizer()
    Y_mechanism = mlb.fit_transform(df_cleaned["Terms_List"])  # multi-label mechanism
    label_columns = mlb.classes_

    le = LabelEncoder()
    polarity_mask = df_cleaned["polarity"].notna()  # mask for rows with polarity
    polarity_labels = df_cleaned.loc[polarity_mask, "polarity"]
    Y_polarity = le.fit_transform(polarity_labels)

    return Y_mechanism, Y_polarity, polarity_mask.values, label_columns, le.classes_

In [6]:
# preprocess dataframe
df_cleaned = preprocess_dataframe(df_raw)

# encode labels
Y_mech, Y_pol, polarity_mask, mech_labels, pol_labels = encode_labels(df_cleaned)

print("Sample count:", len(df_cleaned))
print("Samples with polarity labels:", polarity_mask.sum())
print("Mechanism labels:", list(mech_labels))
print("Polarity labes:", list(pol_labels))
df_cleaned.sample(5)

Sample count: 26250
Samples with polarity labels: 8750
Mechanism labels: ['autoactivation', 'autocatalysis', 'autofeedback', 'autoinduction', 'autoinhibition', 'autokinase', 'autolysis', 'autophosphorylation', 'autoregulation', 'autoubiquitination', 'non-autoregulatory']
Polarity labes: ['negative', 'neutral', 'positive']


,batch_number,Text_Cleaned,Terms,Terms_List,polarity
13685,3,identification mitochondrial thiamin diphospha...,non-autoregulatory,[non-autoregulatory],None
19448,4,comparison francisella tularensis genomes reve...,non-autoregulatory,[non-autoregulatory],None
18952,4,role actin cytoskeletal dynamics activation cy...,non-autoregulatory,[non-autoregulatory],None
12149,3,gga autoinhibition revisited cytosolic adaptor...,autoinhibition,[autoinhibition],negative
24752,5,negative regulation transcription type ii argi...,non-autoregulatory,[non-autoregulatory],None


In [7]:
# data splitting
def split_train_val(df_cleaned, stratify_col="Terms", test_size=0.1, random_state=42):
    class_counts = df_cleaned[stratify_col].value_counts()
    rare_classes = class_counts[class_counts < 2].index

    df_rare = df_cleaned[df_cleaned[stratify_col].isin(rare_classes)].copy()
    df_rest = df_cleaned[~df_cleaned[stratify_col].isin(rare_classes)].copy()

    if df_rest.empty:
        print("⚠️ Only rare classes found. All data assigned to train.")
        return df_cleaned, df_cleaned.iloc[0:0]

    df_train_rest, df_val = train_test_split(
        df_rest, test_size=test_size, random_state=random_state, stratify=df_rest[stratify_col]
    )
    df_train = pd.concat([df_train_rest, df_rare], ignore_index=True)

    return df_train, df_val


In [8]:
# split cleaned dataframe into train and validation
df_train, df_val = split_train_val(df_cleaned)

# encode labels separately for train and val
Y_mech_train, Y_pol_train, mask_train, _, _ = encode_labels(df_train)

if len(df_val) < 1:
    Y_mech_val = Y_pol_val = mask_val = None
else:
    Y_mech_val, Y_pol_val, mask_val, _, _ = encode_labels(df_val)

**1. Dataset & Dataloader**

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

In [10]:
# Load PubMedBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

In [11]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, Y_mechanism, Y_polarity, polarity_mask, tokenizer, max_len=512):
        self.df = df.reset_index(drop=True)
        self.Y_mech = Y_mechanism
        self.Y_pol = Y_polarity
        self.mask = polarity_mask
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Mapping from dataset index to polarity label index
        self.polarity_index_map = {}
        polarity_counter = 0
        for i, has_pol in enumerate(polarity_mask):
            if has_pol:
                self.polarity_index_map[i] = polarity_counter
                polarity_counter += 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "Text_Cleaned"]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),         # [max_len]
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels_mech": torch.tensor(self.Y_mech[idx]).float(), # multi-label mechanism
            "has_polarity": torch.tensor(bool(self.mask[idx])),   # polarity indicator
        }

        if self.mask[idx]:
            polarity_idx = self.polarity_index_map[idx]  # map dataset idx to Y_polarity row
            item["labels_polarity"] = torch.tensor(self.Y_pol[polarity_idx]).long()
        else:
            item["labels_polarity"] = torch.tensor(-1).long()  # dummy if no polarity

        return item

In [12]:
# create training dataset and loader
dataset_train = MultiTaskDataset(df_train, Y_mech_train, Y_pol_train, mask_train, tokenizer, max_len=384)
train_loader = DataLoader(dataset_train, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)

# create validation dataset and loader

dataset_val = MultiTaskDataset(df_val, Y_mech_val, Y_pol_val, mask_val, tokenizer, max_len=384)
val_loader = DataLoader(dataset_val, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

# inspect one batch
batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels_mech shape:", batch["labels_mech"].shape)
print("labels_polarity example:", batch["labels_polarity"][:5].tolist())
print("has_polarity mask example:", batch["has_polarity"][:5].tolist())

input_ids shape: torch.Size([8, 384])
attention_mask shape: torch.Size([8, 384])
labels_mech shape: torch.Size([8, 11])
labels_polarity example: [-1, -1, 0, -1, 2]
has_polarity mask example: [False, False, True, False, True]


**2. Model Class Define**

In [13]:
import torch
import torch.nn as nn
from transformers import AutoModel

In [14]:
class MultiTaskPubMedBERTClassifier(nn.Module):
    def __init__(self, n_mech_labels, n_polarity_classes, dropout_rate=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        self.mech_classifier = nn.Linear(hidden_size, n_mech_labels)
        self.polarity_classifier = nn.Linear(hidden_size, n_polarity_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])  # [CLS]
        mech_logits = self.mech_classifier(cls_output)
        polarity_logits = self.polarity_classifier(cls_output)
        return mech_logits, polarity_logits

In [15]:
# Device Configuration
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device being used: {device}")

Device being used: mps


In [16]:
# instantiate model
model = MultiTaskPubMedBERTClassifier(
    n_mech_labels=len(mech_labels),
    n_polarity_classes=len(pol_labels),
    dropout_rate=0.3
).to(device)

batch = {k: v.to(device) for k, v in batch.items()}

# forward pass with batch
mech_logits, polarity_logits = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print("mech_logits shape:", mech_logits.shape)
print("polarity_logits shape:", polarity_logits.shape)

mech_logits shape: torch.Size([8, 11])
polarity_logits shape: torch.Size([8, 3])


**3.Loss Function & Training Loop Define**

In [17]:
import torch.nn.functional as F

In [18]:
# multi-task loss computation
def compute_multitask_loss(batch, mech_logits, polarity_logits, pos_weight):
    # 机制分类的损失
    labels_mech = batch["labels_mech"].float()
    labels_polarity = batch["labels_polarity"].long()
    mask = batch["has_polarity"].bool()

    # 添加标签平滑
    epsilon = 0.1
    labels_smooth = labels_mech * (1 - epsilon) + epsilon / labels_mech.shape[1]
    
    # 计算机制分类损失
    mech_loss = F.binary_cross_entropy_with_logits(
        mech_logits,
        labels_smooth,
        pos_weight=pos_weight,
        reduction='none'
    )
    
    # 计算每个样本的损失均值
    mech_loss = mech_loss.mean(dim=1)
    
    # 计算极性分类损失
    if mask.any():
        polarity_loss = F.cross_entropy(
            polarity_logits[mask],
            labels_polarity[mask],
            reduction='mean',
            label_smoothing=0.1
        )
    else:
        polarity_loss = torch.tensor(0.0, device=mech_logits.device)

    # 总损失
    total_loss = mech_loss.mean() + polarity_loss
    
    return total_loss, mech_loss.mean().item(), polarity_loss.item()

In [19]:
# compute pos_weight for each mechanism class
pos_counts = np.sum(Y_mech, axis=0)
neg_counts = len(Y_mech) - pos_counts
weight_ratio = neg_counts / np.maximum(pos_counts, 1)

# convert to torch tensor and move to model device
model_device = next(model.parameters()).device
pos_weight = torch.tensor(weight_ratio, dtype=torch.float32).to(model_device)

print("pos_weight shape:", pos_weight.shape)

pos_weight shape: torch.Size([11])


**4. Train Model**

In [20]:
import os
import json
import torch
from tqdm import tqdm
from torch.optim import AdamW
from sklearn.metrics import f1_score, precision_score, recall_score

In [21]:
# multi-label evaluation
def evaluate_batch(y_true, y_pred, threshold=0.5):
    """
    y_true: numpy array of shape [n_samples, n_labels]
    y_pred: raw logits (torch tensor)
    """
    y_pred = torch.sigmoid(y_pred).cpu().numpy()
    y_true = y_true.cpu().numpy()
    y_pred_bin = (y_pred >= threshold).astype(int)

    # compute scores
    micro_f1 = f1_score(y_true, y_pred_bin, average="micro")
    sample_f1 = f1_score(y_true, y_pred_bin, average="samples")
    sample_prec = precision_score(y_true, y_pred_bin, average="samples", zero_division=0)
    sample_rec = recall_score(y_true, y_pred_bin, average="samples", zero_division=0)

    return micro_f1, sample_f1, sample_prec, sample_rec

In [22]:
def evaluate_with_best_thresholds(y_true, y_pred, label_names=None):
    thresholds = {}
    binarized_preds = np.zeros_like(y_pred)

    for i in range(y_pred.shape[1]):
        best_f1 = 0.0
        best_t = 0.5

        for t in np.linspace(0.1, 0.9, 17):
            preds = (y_pred[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        label = label_names[i] if label_names is not None else f"Label_{i}"
        thresholds[label] = round(best_t, 2)
        binarized_preds[:, i] = (y_pred[:, i] >= best_t).astype(int)

    return thresholds, binarized_preds

In [23]:
# validation on val_loader
def validate_model(model, dataloader, pos_weight, device, label_names=None):
    model.eval()
    total_loss = 0
    all_y_true, all_y_pred = [], []
    pred_distributions = []
    label_distributions = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            mech_logits, polarity_logits = model(batch["input_ids"], batch["attention_mask"])
            
            # 记录预测和标签分布
            pred_dist = torch.sigmoid(mech_logits).mean(dim=0)
            label_dist = batch["labels_mech"].float().mean(dim=0)
            pred_distributions.append(pred_dist.cpu())
            label_distributions.append(label_dist.cpu())
            
            loss, loss_mech, loss_pol = compute_multitask_loss(batch, mech_logits, polarity_logits, pos_weight)
            total_loss += loss.item()
            all_y_true.append(batch["labels_mech"].cpu())
            all_y_pred.append(mech_logits.cpu())

    # 打印验证集的预测分布
    avg_pred_dist = torch.stack(pred_distributions).mean(dim=0)
    avg_label_dist = torch.stack(label_distributions).mean(dim=0)
    print("\nValidation predictions distribution:", avg_pred_dist)
    print("Validation labels distribution:", avg_label_dist)

    y_true = torch.cat(all_y_true, dim=0).numpy()
    y_pred = torch.cat(all_y_pred, dim=0).numpy()

    # 移除全零标签
    valid_mask = y_true.sum(axis=1) > 0
    y_true = y_true[valid_mask]
    y_pred = y_pred[valid_mask]

    # 动态阈值选择
    thresholds_dict, binarized_preds = evaluate_with_best_thresholds(y_true, y_pred, label_names)
    
    # 打印每个类别的阈值
    print("\nSelected thresholds for each class:")
    for label, threshold in thresholds_dict.items():
        print(f"{label}: {threshold:.4f}")

    micro_f1, sample_f1, sample_prec, sample_rec = evaluate_batch(
        torch.tensor(y_true), torch.tensor(binarized_preds)
    )

    avg_loss = total_loss / len(dataloader)
    return avg_loss, micro_f1, sample_f1, sample_prec, sample_rec, thresholds_dict

In [24]:
# train model
def train_model(model, dataloader, optimizer, pos_weight, device, n_epochs=5, val_loader=None, patience=3, batch_number=1, label_names=None):
    model.to(device)
    best_score = 0
    no_improve_count = 0
    best_state = None
    best_thresholds = None
    max_grad_norm = 1.0  # 梯度裁剪阈值
    
    for epoch in range(n_epochs):
        print(f"\nEpoch {epoch + 1}/{n_epochs}")
        model.train()
        total_loss, total_mech, total_pol = 0, 0, 0
        all_y_true, all_y_pred = [], []
        epoch_grad_stats = []

        for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Training Epoch {epoch + 1}")):
            batch = {k: v.to(device) for k, v in batch.items()}
            mech_logits, polarity_logits = model(batch["input_ids"], batch["attention_mask"])
            loss, loss_mech, loss_pol = compute_multitask_loss(batch, mech_logits, polarity_logits, pos_weight)

            optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            
            # 监控梯度
            grad_stats = {}
            for name, param in model.named_parameters():
                if param.grad is not None:
                    grad_stats[name] = param.grad.abs().mean().item()
            epoch_grad_stats.append(grad_stats)
            
            optimizer.step()

            # 打印预测和标签分布
            if batch_idx % 50 == 0:  # 每50个batch打印一次
                with torch.no_grad():
                    pred_dist = torch.sigmoid(mech_logits).mean(dim=0)
                    label_dist = batch["labels_mech"].float().mean(dim=0)
                    print("\nPredictions distribution:", pred_dist)
                    print("Labels distribution:", label_dist)

            all_y_true.append(batch["labels_mech"])
            all_y_pred.append(mech_logits.detach())
            total_loss += loss.item()
            total_mech += loss_mech
            total_pol += loss_pol

        # 打印每个epoch的梯度统计信息
        print("\nGradient statistics for this epoch:")
        avg_grads = {k: np.mean([stats[k] for stats in epoch_grad_stats]) for k in epoch_grad_stats[0].keys()}
        for k, v in avg_grads.items():
            print(f"{k}: {v:.6f}")

        y_true = torch.cat(all_y_true, dim=0)
        y_pred = torch.cat(all_y_pred, dim=0)
        micro_f1, sample_f1, sample_prec, sample_rec = evaluate_batch(y_true, y_pred)

        print(f"Train | Loss: {total_loss / len(dataloader):.4f} | Micro F1: {micro_f1:.4f} | Sample F1: {sample_f1:.4f}")

        if val_loader is not None:
            val_loss, val_micro_f1, val_sample_f1, val_prec, val_rec, threshold_dict = validate_model(
                model, val_loader, pos_weight, device, label_names
            )
            print(f"Val   | Loss: {val_loss:.4f} | Micro F1: {val_micro_f1:.4f} | Sample F1: {val_sample_f1:.4f}")
            print(f"       Precision: {val_prec:.4f} | Recall: {val_rec:.4f}")

            if val_micro_f1 > best_score:
                best_score = val_micro_f1
                best_state = model.state_dict()
                best_thresholds = threshold_dict
                no_improve_count = 0
                print("New best model and thresholds saved.")
            else:
                no_improve_count += 1
                print(f"No improvement. Patience: {no_improve_count}/{patience}")

            if no_improve_count >= patience:
                print("Early stopping triggered.")
                break

    if best_state:
        model_path = f"../src/model/best_model_batch_{batch_number}.pt"
        torch.save(best_state, model_path)
        print(f"Best model saved to {model_path}")

    if best_thresholds:
        threshold_path = f"../src/model/best_thresholds_batch_{batch_number}.json"
        with open(threshold_path, "w") as f:
            json.dump(best_thresholds, f, indent=2)
        print(f"Thresholds saved to {threshold_path}")

In [25]:
def train_for_batch(
    batch_number,
    label_names,
    batch_size=16,
    max_len=384,
    learning_rate=1e-4,  # 降低学习率
    weight_decay=0.01,
    optimizer_type="AdamW",
    dropout_rate=0.3,    # 增加dropout
    n_epochs=7,
    patience=3          # 增加patience
):
    print(f"\nStarting training for Batch {batch_number}")

    # 1. 过滤当前批次
    df_batch = df_raw[df_raw["batch_number"] == batch_number].copy()
    df_cleaned = preprocess_dataframe(df_batch)

    # 2. 训练测试分割
    df_train, df_val = split_train_val(df_cleaned)

    # 3. 机制标签
    Y_mech_train, Y_pol_train, mask_train, mech_labels, pol_labels = encode_labels(df_train)
    Y_mech_val, Y_pol_val, mask_val, _, _ = encode_labels(df_val)

    # 4. 加载数据集和DataLoader
    dataset_train = MultiTaskDataset(df_train, Y_mech_train, Y_pol_train, mask_train, tokenizer, max_len=max_len)
    dataset_val = MultiTaskDataset(df_val, Y_mech_val, Y_pol_val, mask_val, tokenizer, max_len=max_len)

    train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    # 5. 模型初始化
    model = MultiTaskPubMedBERTClassifier(
        n_mech_labels=Y_mech_train.shape[1],
        n_polarity_classes=3,
        dropout_rate=dropout_rate
    ).to(device)

    # 6. 优化器
    if optimizer_type == "AdamW":
        optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_type == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_type == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer type: {optimizer_type}")

    # 7. 正样本权重
    pos_counts = np.sum(Y_mech_train, axis=0)
    neg_counts = len(Y_mech_train) - pos_counts
    weight_ratio = neg_counts / np.maximum(pos_counts, 1)
    pos_weight = torch.tensor(weight_ratio, dtype=torch.float32).to(device)

    # 8. 训练模型
    train_model(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        pos_weight=pos_weight,
        device=device,
        n_epochs=n_epochs,
        val_loader=val_loader,
        patience=patience,
        batch_number=batch_number,
        label_names=label_names
    )

In [27]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
_, _, _, mech_labels, _ = encode_labels(df_cleaned)

# start training the model
for batch in range(1, 6):
    train_for_batch(
        batch_number=batch,
        label_names=mech_labels,
        batch_size=16,
        max_len=384,
        learning_rate=2e-5,
        dropout_rate=0.1,
        n_epochs=5,
        patience=3
    )


Starting training for Batch 1

Epoch 1/5


Training Epoch 1:   0%|          | 1/296 [00:02<11:41,  2.38s/it]


Predictions distribution: tensor([0.4097, 0.4947, 0.4099, 0.5535, 0.5970, 0.4450, 0.5132, 0.5989, 0.4449,
        0.4994, 0.5591], device='mps:0')
Labels distribution: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1250, 0.0000,
        0.0625, 0.8125], device='mps:0')


Training Epoch 1:  17%|█▋        | 51/296 [00:44<03:30,  1.16it/s]


Predictions distribution: tensor([0.7250, 0.5640, 0.7619, 0.6675, 0.5024, 0.8726, 0.5133, 0.4758, 0.4965,
        0.5384, 0.4876], device='mps:0')
Labels distribution: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0625, 0.1875, 0.0625,
        0.0000, 0.6875], device='mps:0')


...


Gradient statistics for this epoch:
encoder.embeddings.word_embeddings.weight: 0.000001
encoder.embeddings.position_embeddings.weight: 0.000034
encoder.embeddings.token_type_embeddings.weight: 0.002567
encoder.embeddings.LayerNorm.weight: 0.000321
encoder.embeddings.LayerNorm.bias: 0.000939
encoder.encoder.layer.0.attention.self.query.weight: 0.000016
encoder.encoder.layer.0.attention.self.query.bias: 0.000047
encoder.encoder.layer.0.attention.self.key.weight: 0.000015
encoder.encoder.layer.0.attention.self.key.bias: 0.000000
encoder.encoder.layer.0.attention.self.value.weight: 0.000056
encoder.encoder.layer.0.attention.self.value.bias: 0.000483
encoder.encoder.layer.0.attention.output.dense.weight: 0.000068
encoder.encoder.layer.0.attention.output.dense.bias: 0.000553
encoder.encoder.layer.0.attention.output.LayerNorm.weight: 0.000181
encoder.encoder.layer.0.attention.output.LayerNorm.bias: 0.000304
encoder.encoder.layer.0.intermediate.dense.weight: 0.000021
encoder.encoder.layer.0.i

Training Epoch 3:   0%|          | 1/296 [00:01<07:34,  1.54s/it]


Predictions distribution: tensor([0.7043, 0.2789, 0.7671, 0.3702, 0.2392, 0.8550, 0.4137, 0.3985, 0.2298,
        0.3483, 0.4773], device='mps:0')
Labels distribution: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3125, 0.0625,
        0.0625, 0.5625], device='mps:0')


Training Epoch 3:   3%|▎         | 9/296 [00:15<08:11,  1.71s/it]


KeyboardInterrupt: 

**5. Test Model**

In [41]:
import torch
import json
import pandas as pd
from transformers import AutoTokenizer

In [46]:
df_test = pd.read_csv('../data/processed/test_data.csv')
print(df_test.shape)
df_test.sample(5)

(38, 8)


,AC,PMID,Title,Abstract,if_contain_keyterm,Terms,Text_combined,Polarity
5,P9WEN4,31117659,Distinct Autocatalytic α- N-Methylating Precur...,Backbone N-methylations impart several favorab...,0,autocatalysis,Distinct Autocatalytic - N-Methylating Precur...,NaN
9,Q99986,21543316,NMR solution structure of human vaccinia-relat...,Vaccinia-related kinase 1 (VRK1) is one of the...,0,"autocatalysis, autophosphorylation",NMR solution structure of human vaccinia-relat...,NaN
33,[AI-generated],AI_14,NaN,GAIN domain-mediated self-cleavage is constitu...,0,autocatalysis,GAIN domain-mediated self-cleavage is constit...,positive
20,[AI-generated],AI_1,NaN,The transcription factor binds to its own prom...,0,autoregulation,The transcription factor binds to its own pro...,neutral
12,P12979,1324403,Analysis of the myogenin promoter reveals an i...,Transcriptional cascades that specify cell fat...,1,autoregulation,Analysis of the myogenin promoter reveals an i...,NaN


In [49]:
def predict_on_test_data(batch_number, model_path, threshold_path, label_names, max_len=384, base_df=None):
    """
    Predict mechanism terms and polarity on the test set for a specific batch.
    If base_df is provided, it will append predictions to that DataFrame.
    """
    print(f"\n🚀 Running prediction for Batch {batch_number} ...")

    # 1. Load test set or use existing DataFrame
    if base_df is None:
        df = pd.read_csv("../data/processed/test_data.csv").copy()
        df["Actual Term"] = df["Terms"]
        df["Actual Polarity"] = df["Polarity"] if "Polarity" in df.columns else None
    else:
        df = base_df.copy()

    # 2. Load model & tokenizer
    tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")
    model = MultiTaskPubMedBERTClassifier(
        n_mech_labels=len(label_names),
        n_polarity_classes=3
    )
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()

    # 3. Load threshold
    with open(threshold_path, "r") as f:
        thresholds = json.load(f)

    # 4. Tokenize
    texts = df["Text_combined"].astype(str).tolist()
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    # 5. Predict
    with torch.no_grad():
        mech_logits, polarity_logits = model(inputs["input_ids"], inputs["attention_mask"])
        mech_probs = torch.sigmoid(mech_logits).numpy()
        polarity_probs = torch.softmax(polarity_logits, dim=1).numpy()

    # 6. Apply threshold
    pred_mechs = []
    pred_pols = []

    for i in range(len(df)):
        pred_labels = [
            label for j, label in enumerate(label_names)
            if mech_probs[i][j] >= thresholds.get(label, 0.5)
        ]
        pred_mechs.append(pred_labels)

        if pred_labels:
            polarity_idx = polarity_probs[i].argmax()
            polarity_label = ["negative", "neutral", "positive"][polarity_idx]
        else:
            polarity_label = None

        pred_pols.append(polarity_label)

    # 7. Store into specific columns
    df[f"Pred Term Batch {batch_number}"] = pred_mechs
    df[f"Pred Polarity Batch {batch_number}"] = pred_pols

    # 8. Return only relevant columns
    base_cols = ["AC", "PMID", "Actual Term", "Actual Polarity"]
    pred_cols = [f"Pred Term Batch {batch_number}", f"Pred Polarity Batch {batch_number}"]
    return df[base_cols + pred_cols]


In [ ]:
df_pred = pd.read_csv("../data/processed/test_data.csv")
df_pred["Actual Term"] = df_pred["Terms"]
df_pred["Actual Polarity"] = df_pred["Polarity"]

# test all batches
for b in range(1, 2):
    df_pred = predict_on_test_data(
        batch_number=b,
        model_path=f"../src/model/best_model_batch_{b}.pt",
        threshold_path=f"../src/model/best_thresholds_batch_{b}.json",
        label_names=mech_labels,
        base_df=df_pred
    )


🚀 Running prediction for Batch 1 ...


/var/folders/1h/csb8qjzj1sv0jtrd6ccllm8c0000gn/T/ipykernel_98768/3162095906.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path,

In [51]:
df_pred

,AC,PMID,Actual Term,Actual Polarity,Pred Term Batch 1,Pred Polarity Batch 1
0,P00519,16543148,autoinhibition,NaN,"[autoactivation, autocatalysis, autoinhibition...",negative
1,Q8RXD3,15998807,autoubiquitination,NaN,"[autoinhibition, autokinase, autophosphorylati...",negative
2,P00520,20072125,autophosphorylation,NaN,"[autoactivation, autocatalysis, autoinhibition...",negative
3,B0FLN1,18281398,autoinduction,NaN,"[autoinduction, autolysis, autoregulation]",positive
4,Q9Y4W6,37917749,autoregulation,NaN,"[autoinhibition, autokinase, autoregulation, a...",negative
5,P9WEN4,31117659,autocatalysis,NaN,"[autoactivation, autocatalysis, autoinhibition...",negative
6,Q2G2U4,17827301,autolysis,NaN,"[autocatalysis, autoinduction, autokinase, aut...",positive
7,P06213,12707268,"autoinhibition, autophosphorylation",NaN,"[autoactivation, autocatalysis, autoinhibition...",negative
8,Q06486,7665585,"autoinhibition, autophosphorylation",NaN,"[autoactivation, autocatalysis, autoinhibition...",positive
9,Q99986,21543316,"autocatalysis, autophosphorylation",NaN,"[autoactivation, autocatalysis, autoinhibition...",negative
